# Module 1 : 70 ans d'IA en 30 minutes

*Mardi 23 juin 2026, 9h-12h · Intervenant : Corentin Vande Kerckhove*

**CONSEIL :** Commencez par enregistrer une copie de ce notebook dans votre Google Drive (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes !

**Objectifs d'apprentissage**
- Situer l'IA générative dans l'histoire de l'IA (symbolique → ML → DL → Transformers → LLM)
- Voir la **même tâche** (classer un texte par parti politique) traverser chaque époque
- Comprendre ce que chaque génération ajoute à la précédente
- Voir **un modèle apprendre** depuis des données, et lire ce qu'il a appris
- Apprivoiser un premier appel à un grand modèle de langage

**Pré-requis :** aucun, c'est le premier module de la formation.
**Paquets requis :** `scikit-learn`, `datasets`, `transformers`, `anthropic`, `pandas`

## Running example : un peu de politique française

On va se balader dans 70 ans d'IA avec une question fil rouge : **« à quel parti appartient l'auteur de cette déclaration ? »**.

Le RN aime parler d'immigration. LFI préfère les salaires et le peuple. LR adore le marché libre et les entreprises. EELV milite pour la transition écologique et le climat. Vous voyez l'idée.

> Pour illustrer les concepts, on travaille sur un **mini-corpus de 16 phrases inventées** (4 par parti). C'est volontairement petit : assez pour voir 70 ans d'IA défiler, et sans manipuler de données personnelles réelles.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j2/1_matin/img/m1-opener.jpg" width="440" alt="Urne de vote">

*Fil rouge du module : classer des déclarations politiques par parti. Photo : urne de vote, FutUndBeidl, CC BY 2.0.*

## Setup

In [ ]:
# Sur Colab : exécute cette cellule pour installer les paquets (déjà installés sur un poste local)
!pip install -q scikit-learn datasets transformers anthropic pandas

import pandas as pd

In [ ]:
# 16 déclarations courtes, 4 par parti
data = pd.DataFrame({
    "texte": [
        # RN
        "Il faut réduire l'immigration et défendre nos frontières.",
        "L'identité française doit être protégée.",
        "Sécurité, frontières, immigration : nos priorités sont claires.",
        "Notre identité et nos traditions face à la submersion migratoire.",
        # LFI
        "Augmentons les salaires et taxons les plus riches.",
        "Le peuple souffre pendant que la finance s'enrichit.",
        "Une révolution citoyenne contre l'oligarchie est nécessaire.",
        "Taxons les riches pour financer les services publics et les salaires.",
        # LR
        "Le marché libre et la baisse des charges relanceront l'économie.",
        "Soutenons les entreprises et libérons les énergies.",
        "Moins d'État, plus de responsabilité et de liberté économique.",
        "Baissons les charges des entreprises pour relancer le marché du travail.",
        # EELV
        "La transition écologique est urgente et juste.",
        "Le climat et la biodiversité au cœur des politiques publiques.",
        "Sortons des énergies fossiles, investissons dans le renouvelable.",
        "Une transition écologique juste pour préserver le climat.",
    ],
    "parti": ["RN", "RN", "RN", "RN",
              "LFI", "LFI", "LFI", "LFI",
              "LR", "LR", "LR", "LR",
              "EELV", "EELV", "EELV", "EELV"],
})
data

## 1.1 IA symbolique (années 50-80)

Nos ancêtres en IA n'avaient ni Internet, ni puissance de calcul, ni montagnes de données.
Leur arme : **des règles écrites à la main** par un expert humain.

Concrètement : pour reconnaître un texte du RN, on dresse une liste de mots qui reviennent souvent chez eux (« immigration », « frontière »…). Idem pour les autres partis. Puis on compte, pour chaque texte, combien de mots-clés de chaque parti apparaissent. Le parti qui en a le plus gagne. Voilà notre première « IA ».

**Force** : c'est lisible, on sait exactement ce que le programme fait.
**Limite** : c'est très fragile. Une métaphore ou une formule détournée, et tout casse.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j2/1_matin/img/m1-symbolic.png" width="560" alt="Schéma IA symbolique">

*L'humain écrit des règles de mots-clés ; la machine se contente de compter.*

In [ ]:
# Une mini "IA" : pour chaque parti, on liste des mots-clés.
RULES = {
    "RN":   ["immigration", "frontière", "identité"],
    "LFI":  ["salaire", "riche", "peuple"],
    "LR":   ["marché", "libre", "charges", "entreprise"],
    "EELV": ["écolog", "transition", "climat"],
}

def compter_mots_cles(texte):
    """Pour chaque parti, compter combien de ses mots-clés apparaissent dans le texte."""
    texte = texte.lower()
    return {parti: sum(m in texte for m in mots) for parti, mots in RULES.items()}

def predire_par_regles(texte):
    """Renvoyer le parti qui obtient le plus grand score (ou '?' si tout le monde a 0)."""
    scores = compter_mots_cles(texte)
    if max(scores.values()) == 0:
        return "?"
    return max(scores, key=scores.get)

# On regarde à la fois les scores détaillés et la prédiction finale
data["scores_rules"] = data["texte"].apply(compter_mots_cles)
data["pred_rules"]   = data["texte"].apply(predire_par_regles)
data[["parti", "scores_rules", "pred_rules"]].head(10)

### Hack Time

- Ajoutez un mot-clé par parti dans le dictionnaire `RULES`
- Comptez le nombre de prédictions correctes (`(data["parti"] == data["pred_rules"]).sum()`)
- Essayez d'écrire une phrase qui « casse » votre classifier (ironie, métaphore, mot d'un autre parti)

In [ ]:
# Votre code ici

## 1.2 Machine Learning statistique (années 90-2010)

L'idée centrale du ML : **arrêter d'écrire les règles à la main, et les apprendre depuis les données**.

Reformulons : en 1.1, c'est nous qui décidions que tel mot-clé compte pour tel parti. Le programme se contentait d'appliquer notre liste, il n'apprend rien tout seul.

En ML, on procède plutôt en trois temps :
1. On **prépare des features** simples (par exemple : « le mot 'immigration' est-il présent ? »)
2. On donne au modèle des **exemples étiquetés** (les 16 phrases ci-dessus)
3. On laisse le modèle découvrir tout seul **combien chaque mot compte pour chaque parti**

Le modèle qu'on va utiliser est la **régression logistique**, un cousin proche de la régression linéaire que beaucoup d'entre vous ont déjà rencontrée. Avantage : à la fin, on peut **lire ses coefficients** comme on lirait les coefficients d'une régression classique.

### 1.2.1 Préparer les features

In [ ]:
# Pour chaque mot-clé, on crée une colonne 0/1 : "ce mot apparaît-il dans le texte ?"
MOTS_CLES = ["immigration", "frontière", "identité",
             "salaire", "riche", "peuple",
             "marché", "libre", "charge", "entreprise",
             "écolog", "transition", "climat"]

def extraire_features(texte):
    texte = texte.lower()
    return {mot: int(mot in texte) for mot in MOTS_CLES}

features = pd.DataFrame([extraire_features(t) for t in data["texte"]])
features.insert(0, "parti", data["parti"])
features

Vous obtenez une **matrice de features** : chaque ligne = une phrase, chaque colonne = la présence (1) ou l'absence (0) d'un mot-clé.

C'est sur cette matrice que le modèle va travailler. Plus de texte brut, juste des 0 et des 1.

### 1.2.2 Voir le modèle apprendre

In [ ]:
from sklearn.linear_model import LogisticRegression

# Variables d'entrée et de sortie pour le modèle
X = pd.DataFrame([extraire_features(t) for t in data["texte"]])
y = data["parti"]

# Le modèle vide, qui ne sait encore rien
modele = LogisticRegression(max_iter=1000)

# Il apprend depuis les 16 phrases (et leur étiquette de parti)
modele.fit(X, y)

# On lit ce qu'il a appris : ses coefficients.
# Une ligne par parti, une colonne par mot-clé.
# Plus le coefficient est élevé, plus le mot tire vers ce parti.
coefs = pd.DataFrame(
    modele.coef_,
    columns=MOTS_CLES,
    index=modele.classes_,
).round(2)
coefs

**Lisez attentivement cette table.** Le modèle a découvert tout seul, sans qu'on lui dise rien :
- que `immigration` et `frontière` poussent vers RN,
- que `salaire` et `riche` poussent vers LFI,
- que `marché` et `entreprise` poussent vers LR,
- que `transition` et `climat` poussent vers EELV.

Bref, **il a réinventé nos règles de la section 1.1, mais en chiffres**, et avec des nuances en plus.

### 1.2.3 Faire prédire le modèle sur une phrase de votre cru

In [ ]:
ma_phrase = "Il faut taxer les multinationales pour financer l'hôpital public."

mes_features = pd.DataFrame([extraire_features(ma_phrase)])
prediction = modele.predict(mes_features)[0]
probabilites = pd.Series(modele.predict_proba(mes_features)[0], index=modele.classes_).round(2)

print(f"Prédiction : {prediction}")
print()
print("Probabilités estimées par le modèle :")
print(probabilites)

### Hack Time

Trois pistes pour explorer le comportement du modèle :

- **Modifiez `ma_phrase`** pour faire dire chaque parti au modèle. Essayez une phrase **sans aucun mot-clé** de la liste : que prédit-il et avec quelle confiance ?
- **Ajoutez une 17ᵉ phrase** à `data` (par exemple un message LFI qui parle aussi d'écologie), relancez tout ce qui précède : les coefficients changent-ils beaucoup ?
- **Retirez un mot-clé** de `MOTS_CLES` et relancez : sur quel autre mot le modèle se rabat-il pour repérer le parti concerné ?

## 1.3 Deep Learning (2010-2017)

Bascule majeure : on **arrête de fabriquer les features à la main**.

Au lieu de notre liste `MOTS_CLES` (qu'on avait choisie nous-mêmes), on laisse un réseau de neurones apprendre lui-même une représentation des textes (les fameux **embeddings**, qu'on verra en J3 text mining).

Pour notre tâche, c'est plus complexe à mettre en place mais ça commence à mieux marcher, surtout quand on a beaucoup de données.

> **Pour voir un réseau de neurones apprendre en direct** : ouvrez [playground.tensorflow.org](https://playground.tensorflow.org/) dans un nouvel onglet. Choisissez un dataset (en haut à gauche), cliquez sur Play, et regardez les couches « apprendre » à séparer les points en quelques secondes. C'est la même mécanique sous le capot des LLMs, en beaucoup plus gros.

### 1.3.1 Un mini-réseau de neurones sur nos features

Pour rester comparable avec la section 1.2, on garde les mêmes features (présence/absence binaire des mots-clés), mais on remplace la régression logistique par un **réseau de neurones simple** (un *multilayer perceptron* : plusieurs couches de neurones empilées).

Pourquoi essayer ça ? Parce qu'un réseau peut apprendre des **combinaisons non-linéaires** des mots-clés. Là où la régression logistique additionne les contributions de chaque mot, le réseau peut détecter des configurations plus subtiles.

In [ ]:
from sklearn.neural_network import MLPClassifier

# Un petit réseau à 2 couches cachées (16 puis 8 neurones)
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=2000, random_state=42)
mlp.fit(X, y)
print("Précision sur les 16 phrases :", round(mlp.score(X, y), 2))

Notez que le réseau atteint la même précision que la régression logistique sur les phrases d'entraînement. Sur ce petit corpus très propre, il n'a pas vraiment besoin de sa capacité non-linéaire : il s'aligne sur les mêmes signaux que la régression.

### Hack Time

- Changez un hyperparamètre du réseau (par exemple `hidden_layer_sizes` ou `max_iter`), relancez l'entraînement et observez si la précision et les prédictions bougent.

**Défi optionnel :** réduisez le réseau à une seule couche minuscule (`hidden_layer_sizes=(2,)`) et voyez à partir de quand il n'arrive plus à apprendre nos 16 phrases.

In [ ]:
# Votre code ici

## 1.4 Transformers (2017→)

**2017 : *Attention is all you need***. Une nouvelle architecture, le **transformer**, débarque chez Google.

Pourquoi ça change tout ? Parce que pour la première fois, un modèle peut être pré-entraîné sur d'énormes quantités de texte, **comprendre le sens des mots dans leur contexte**, et être ensuite adapté à n'importe quelle tâche avec très peu d'exemples.

L'innovation clé : le **mécanisme d'attention**. Chaque mot regarde tous les autres pour saisir son rôle exact dans la phrase. (Schéma en slide.)

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j2/1_matin/img/m1-transformers.png" width="560" alt="Schéma mécanisme d'attention">

*L'attention : chaque mot fixe son sens en regardant les autres mots de la phrase.*

### 1.4.1 Charger un transformer pré-entraîné

On quitte ici scikit-learn, où nous fabriquions nous-mêmes les features, pour la librairie `transformers`, qui fournit des modèles déjà pré-entraînés sur d'immenses corpus de texte.

On va utiliser un transformer **général** déjà entraîné sur d'énormes corpus multilingues, sans aucun entraînement supplémentaire sur nos partis politiques. C'est ce qu'on appelle un usage **zero-shot** : on demande au modèle de choisir une étiquette parmi une liste qu'on lui donne, et lui se débrouille avec ses connaissances générales du langage.

In [ ]:
# Charger le transformer (peut prendre 30-60 s la première fois, le temps du téléchargement)
from transformers import pipeline

clf_transformer = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
)

# Première utilisation : on lui donne une phrase et la liste des étiquettes possibles
clf_transformer(
    "Augmentons les salaires et taxons les plus riches.",
    candidate_labels=["RN", "LFI", "LR", "EELV"],
)

Le résultat est un dictionnaire qui donne pour chaque étiquette une probabilité. La première dans la liste `labels` est la prédiction du modèle.

Pour s'en servir comme classifier, on enveloppe ça dans une petite fonction :

In [ ]:
def classer_transformer(texte):
    return clf_transformer(texte, candidate_labels=["RN", "LFI", "LR", "EELV"])["labels"][0]

# Test rapide sur une de nos phrases
classer_transformer("La transition écologique est urgente et juste.")

### Hack Time

- Remplacez les `candidate_labels` donnés au classifier zero-shot (par exemple par des libellés plus parlants comme `"extrême droite"`, `"gauche radicale"`, `"droite libérale"`, `"écologistes"`) et relancez `clf_transformer` sur une phrase de test : la prédiction change-t-elle ?

**Défi optionnel :** appliquez `classer_transformer` aux 4 phrases pièges de la section 1.5 et comparez avec les autres méthodes.

In [ ]:
# Votre code ici

## 1.5 Comparaison des 4 modèles

On a maintenant 4 façons de classer un texte par parti. Question : **est-ce que chaque génération est vraiment meilleure que la précédente ?**

Pour le savoir, on va leur poser un test piégeux : 4 phrases qui paraphrasent les thèmes des partis **mais sans utiliser aucun de nos mots-clés**. Synonymes uniquement.

In [ ]:
phrases_pieges = pd.DataFrame({
    "texte": [
        "L'arrivée massive d'étrangers doit être limitée.",        # thème RN, sans "immigration"
        "Augmentons la rémunération des plus modestes.",           # thème LFI, sans "salaire"
        "Encourageons les sociétés et l'esprit d'initiative.",     # thème LR, sans "entreprise"
        "Préservons notre planète et nos écosystèmes.",            # thème EELV, sans "écolog"
    ],
    "parti_vrai": ["RN", "LFI", "LR", "EELV"],
})
phrases_pieges

### 1.5.1 Lancer les 4 modèles côte à côte

In [ ]:
# Features binaires (mêmes que les sections 1.2 et 1.3)
feats_pieges = pd.DataFrame([extraire_features(t) for t in phrases_pieges["texte"]])

# Les 3 premières méthodes utilisent les mots-clés comme entrée
resultats = phrases_pieges.copy()
resultats["① règles"]    = resultats["texte"].apply(predire_par_regles)
resultats["② log_reg"]   = modele.predict(feats_pieges)
resultats["③ mlp"]       = mlp.predict(feats_pieges)

# ④ Le transformer travaille directement sur le texte brut
# resultats["④ transformer"] = resultats["texte"].apply(classer_transformer)

resultats

### 1.5.2 Lecture du tableau

| | Méthode | Entrée | Pourquoi ça réussit ou pas |
|---|---|---|---|
| ① | Règles à la main | mots-clés | Aucun mot-clé dans les phrases → renvoie `?` |
| ② | Régression logistique | mots-clés | Idem : aucun signal en entrée → prédiction aléatoire |
| ③ | Réseau de neurones | mots-clés | Idem encore : un modèle plus complexe sur la même entrée vide ne fait pas mieux |
| ④ | Transformer | texte brut | Reconnaît que « rémunération » ≈ « salaire », « écosystème » ≈ « écologie », etc. |

**Le saut conceptuel.** Augmenter la complexité du modèle (de la régression au MLP) ne sauve pas la mise quand les **features d'entrée** sont limitées à une poignée de mots-clés. Le transformer change la donne parce qu'il **ne dépend plus de nos mots-clés** : il lit directement le texte avec un bagage linguistique pré-acquis.

C'est exactement cette capacité à **comprendre le sens indépendamment des mots exacts** qui a ouvert la voie à l'IA générative qu'on voit en 1.6.

### Hack Time

- Inventez une 5ᵉ phrase piégeuse pour un parti de votre choix (sans utiliser nos mots-clés). Lesquelles des 4 méthodes tombent en panne ?
- Bonus : ajoutez 2-3 synonymes (par exemple `étranger` à côté de `immigration`) dans `MOTS_CLES` et relancez les sections 1.2 → 1.5. À partir de quel moment les méthodes ① ② ③ commencent-elles à rattraper le transformer ?

**Défi optionnel :** activez la colonne `④ transformer` (décommentez la ligne dans la cellule ci-dessus) et comparez les 4 méthodes sur les phrases pièges.

## 1.6 IA générative (2020 → aujourd'hui)

Le transformer de 1.4 « comprend » le texte. Mais il faut une dernière bascule pour qu'il **génère** du texte : on l'utilise dans une variante **décodeur** (le modèle ne voit que les mots à sa gauche et apprend à prédire le suivant).

C'est ce qui a donné ChatGPT, Claude, Mistral, Gemini : le modèle ne se contente plus de classer, il **génère sa réponse en langage naturel** et peut expliquer son choix.

Magique ? Pas vraiment. Sous le capot, c'est toujours du transformer. Mais la combinaison **échelle massive + affinage par retours humains** rend le résultat saisissant.

> Note : on dialogue avec un grand modèle de langage via une **API**, c'est-à-dire un service en ligne auquel on envoie du texte et qui nous renvoie une réponse. Pour s'authentifier, on utilise une « clé d'API » (un long mot de passe technique) qu'on ne code jamais en dur dans le notebook. Sur Colab, vous la stockez dans les « Secrets » (l'icône en forme de clé dans la barre latérale gauche) sous le nom `ANTHROPIC_API_KEY`, et le code la relit avec `os.getenv("ANTHROPIC_API_KEY")`.

> **Esprit critique.** La réponse d'un LLM n'est pas une source de vérité : il peut halluciner, il n'est pas parfaitement reproductible (la même question peut donner des réponses différentes), et ses sorties doivent être validées, exactement comme n'importe quel instrument de recherche.

<img src="https://raw.githubusercontent.com/mickaeltemporao/lillelms/refine-notebooks-j2-j3-matin/ressources/j2/1_matin/img/m1-generative.png" width="560" alt="Schéma génération mot à mot">

*Un modèle génératif prédit le mot suivant selon des probabilités, puis recommence.*

In [ ]:
import os
from anthropic import Anthropic

# La clé est lue depuis l'environnement (sur Colab : les "Secrets", icône clé à gauche)
api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    print("Pas de clé ANTHROPIC_API_KEY détectée.")
    print("Ajoute-la dans les Secrets Colab (icône clé) sous le nom ANTHROPIC_API_KEY,")
    print("puis relance cette cellule pour interroger le modèle en direct.")
else:
    client = Anthropic(api_key=api_key)
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": "Classe cette déclaration parmi {RN, LFI, LR, EELV} "
                       "et justifie en une phrase : "
                       "'Augmentons les salaires et taxons les plus riches.'"
        }],
    )
    print(msg.content[0].text)

### Hack Time

Une fois la cellule exécutée :
- Relancez 3 fois la même requête. Obtenez-vous toujours la même réponse ? (Indice : non.)
- Essayez de faire dire au modèle qu'une phrase RN est en fait du LFI. Qui gagne, vous ou le modèle ?
- Mesurez intuitivement : sur 10 phrases, combien le modèle classerait correctement ? Comparez à votre intuition pour les règles de 1.1 et à la régression logistique de 1.2.

**Défi optionnel :** demandez au modèle de justifier sa réponse, puis vérifiez si sa justification tient la route ou s'il invente.

## Récap module 1

| Section | Approche | Forces | Limites |
|---|---|---|---|
| 1.1 IA symbolique | Règles à la main + comptage | Lisible, contrôlable | Tombe sur les synonymes |
| 1.2 ML statistique | Features à la main + régression logistique apprise | Interprétable, des poids depuis les données | Limitée aux mots-clés choisis |
| 1.3 Deep Learning | Réseau de neurones (MLP) sur les mêmes features | Capture des combinaisons non-linéaires | Toujours limité par les features d'entrée |
| 1.4 Transformer | Pré-entraînement massif + attention | Comprend le sens, gère les paraphrases | Boîte noire |
| 1.5 Comparaison | Les 4 ci-dessus mis en concurrence | Rend visible la progression | Demande un dataset bien choisi |
| 1.6 LLM génératif | Le transformer en mode décodeur | Génération libre + explication | Hallucinations, coût |

→ Direction le **module 2** : les concepts ML (features, split, overfitting, biais-variance) qui structurent **tout** ça, y compris ce qui se passe dans le ventre d'un LLM.